In [2]:
import subprocess
import os

LOCAL_BRONZE = r"C:\Users\smagu\retail-data-platform\data\bronze"
os.makedirs(LOCAL_BRONZE, exist_ok=True)

# Télécharger les CSV depuis HDFS vers ta machine
tables = ["clients", "employes", "fournisseurs", "produits", "ventes"]

for table in tables:
    subprocess.run([
        "docker", "exec", "namenode",
        "hdfs", "dfs", "-get", "-f",
        f"/data/bronze/{table}/{table}.csv",
        f"/tmp/{table}.csv"
    ])
    subprocess.run([
        "docker", "cp",
        f"namenode:/tmp/{table}.csv",
        f"{LOCAL_BRONZE}\\{table}.csv"
    ])
    print(f"Téléchargé : {table}.csv")

print("Tous les CSV sont en local !")

Téléchargé : clients.csv
Téléchargé : employes.csv
Téléchargé : fournisseurs.csv
Téléchargé : produits.csv
Téléchargé : ventes.csv
Tous les CSV sont en local !


In [2]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Session SANS config HDFS
spark = SparkSession.builder \
    .appName("03_Bronze_to_Silver") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)

# Lire les CSV locaux
LOCAL = "C:/Users/smagu/retail-data-platform/data/bronze"

clients      = spark.read.csv(f"{LOCAL}/clients.csv",      header=True, inferSchema=True)
employes     = spark.read.csv(f"{LOCAL}/employes.csv",     header=True, inferSchema=True)
fournisseurs = spark.read.csv(f"{LOCAL}/fournisseurs.csv", header=True, inferSchema=True)
produits     = spark.read.csv(f"{LOCAL}/produits.csv",     header=True, inferSchema=True)
ventes       = spark.read.csv(f"{LOCAL}/ventes.csv",       header=True, inferSchema=True)

print("Donnees chargees !")
ventes.show(5)

Spark OK : 3.5.1
Donnees chargees !
+-------+----------+--------+---------+---------+--------------+------------+
|VenteID| DateVente|ClientID|ProduitID|EmployeID|QuantiteVendue|MontantTotal|
+-------+----------+--------+---------+---------+--------------+------------+
|      1|2021-04-09|      67|       73|       11|           550|    439450.0|
|      2|2022-03-24|      42|       87|       65|         19499| 1.9479501E7|
|      3|2023-03-07|      26|       62|       88|          4399|   1315301.0|
|      4|2021-05-26|      32|       60|       23|          1840|   1286160.0|
|      5|2020-01-07|      64|        4|        8|          2400|    717600.0|
+-------+----------+--------+---------+---------+--------------+------------+
only showing top 5 rows



In [3]:
produits.show(5)


+---------+------------------+--------------------+------------+-------------+
|ProduitID|        NomProduit|         Description|PrixUnitaire|FournisseurID|
+---------+------------------+--------------------+------------+-------------+
|        1|Samsung Galaxy S21|Featuring a 6.2-i...|       699.0|            4|
|        2|    Samsung TV 55'|55-inch 4K UHD Sm...|       129.0|           51|
|        3|      Nike Air Max|Running shoes wit...|        85.0|           93|
|        4|      Levi's Jeans|Classic denim jea...|       299.0|            8|
|        5|      Dyson Vacuum|Cordless vacuum c...|       999.0|           58|
+---------+------------------+--------------------+------------+-------------+
only showing top 5 rows



In [3]:
print("=== NULLS PAR TABLE ===")
from pyspark.sql.functions import isnull, sum as spark_sum

for nom, df in [("clients",clients), ("employes",employes),
                ("fournisseurs",fournisseurs), ("produits",produits),
                ("ventes",ventes)]:
    print(f"\n--- {nom.upper()} ---")
    df.select([
        spark_sum(isnull(c).cast("int")).alias(c)
        for c in df.columns
    ]).show()

=== NULLS PAR TABLE ===

--- CLIENTS ---
+--------+---+------+-------+-----+---------------+
|ClientID|Nom|Prenom|Adresse|Email|NumeroTelephone|
+--------+---+------+-------+-----+---------------+
|       0|  0|     0|      0|    0|              0|
+--------+---+------+-------+-----+---------------+


--- EMPLOYES ---
+---------+---+------+--------+-----+---------------+
|EmployeID|Nom|Prenom|Fonction|Email|NuméroTelephone|
+---------+---+------+--------+-----+---------------+
|        0|  0|     0|       0|    0|              0|
+---------+---+------+--------+-----+---------------+


--- FOURNISSEURS ---
+-------------+--------------+-------+-----+---------------+
|FournisseurID|NomFournisseur|Adresse|Email|NumeroTelephone|
+-------------+--------------+-------+-----+---------------+
|            0|             0|      0|    0|              0|
+-------------+--------------+-------+-----+---------------+


--- PRODUITS ---
+---------+----------+-----------+------------+-------------+
|

In [4]:
print("=== CLIENTS ===")
clients_silver = clients \
    .dropDuplicates(["ClientID"]) \
    .dropna(subset=["ClientID", "Nom"]) \
    .withColumn("Nom",    F.trim(F.upper(F.col("Nom")))) \
    .withColumn("Prenom", F.trim(F.initcap(F.col("Prenom")))) \
    .withColumn("Email",  F.lower(F.trim(F.col("Email"))))

print(f"Bronze : {clients.count()} lignes")
print(f"Silver : {clients_silver.count()} lignes")
clients_silver.show(5, truncate=False)

=== CLIENTS ===
Bronze : 100 lignes
Silver : 100 lignes
+--------+-------+------+------------------+----------------------+---------------+
|ClientID|Nom    |Prenom|Adresse           |Email                 |NumeroTelephone|
+--------+-------+------+------------------+----------------------+---------------+
|1       |DOE    |John  |123 Rue de la Demo|john.doe@email.com    |1234567890     |
|2       |SMITH  |Alice |456 Avenue Exemple|alice.smith@email.com |9876543210     |
|3       |JOHNSON|Bob   |789 Rue Test      |bob.johnson@email.com |5551234567     |
|4       |GARCIA |Maria |321 Elm Street    |maria.garcia@email.com|9998887777     |
|5       |CHEN   |Li    |555 Oak Avenue    |li.chen@email.com     |1112223333     |
+--------+-------+------+------------------+----------------------+---------------+
only showing top 5 rows



In [4]:
print("=== EMPLOYES ===")
employes_silver = employes \
    .dropDuplicates(["EmployeID"]) \
    .dropna(subset=["EmployeID", "Nom"]) \
    .withColumn("Nom",    F.trim(F.upper(F.col("Nom")))) \
    .withColumn("Prenom", F.trim(F.initcap(F.col("Prenom")))) \
    .withColumn("Email",  F.lower(F.trim(F.col("Email"))))

print(f"Bronze : {employes.count()} lignes")
print(f"Silver : {employes_silver.count()} lignes")
employes_silver.show(5, truncate=False)

=== EMPLOYES ===
Bronze : 100 lignes
Silver : 100 lignes
+---------+---------+--------+-------------------------------+---------------------------+---------------+
|EmployeID|Nom      |Prenom  |Fonction                       |Email                      |NuméroTelephone|
+---------+---------+--------+-------------------------------+---------------------------+---------------+
|1        |CHANG    |Victoria|Customer Service Representative|chang.victoria@gmail.com   |860084281      |
|2        |MCCOY    |James   |Product Manager                |mccoy.james@gmail.com      |454073892      |
|3        |ADAMS    |Mary    |Administrative Assistant       |adams.mary@gmail.com       |423668243      |
|4        |MCCORMICK|Kirsten |Administrative Assistant       |mccormick.kirsten@gmail.com|4406094        |
|5        |BOWMAN   |Monica  |Sales Representative           |bowman.monica@gmail.com    |242491248      |
+---------+---------+--------+-------------------------------+-------------------------

In [5]:
print("=== FOURNISSEURS ===")
fournisseurs_silver = fournisseurs \
    .dropDuplicates(["FournisseurID"]) \
    .dropna(subset=["FournisseurID", "NomFournisseur"]) \
    .withColumn("NomFournisseur", F.trim(F.upper(F.col("NomFournisseur")))) \
    .withColumn("Email",          F.lower(F.trim(F.col("Email"))))

print(f"Bronze : {fournisseurs.count()} lignes")
print(f"Silver : {fournisseurs_silver.count()} lignes")
fournisseurs_silver.show(5, truncate=False)

=== FOURNISSEURS ===
Bronze : 100 lignes
Silver : 100 lignes
+-------------+-------------------------+-----------------------------------------------------+--------------------------------+---------------+
|FournisseurID|NomFournisseur           |Adresse                                              |Email                           |NumeroTelephone|
+-------------+-------------------------+-----------------------------------------------------+--------------------------------+---------------+
|1            |NELSON, SMITH AND JACKSON|74883 Kimberly Shore, Stewartmouth, VA 11290         |nelson_smithandjackson@gmail.com|399791118      |
|2            |JONES, HODGE AND BURGESS |32014 Andrew Tunnel Suite 839, Campbellport, HI 81373|jones_hodgeandburgess@gmail.com |822754147      |
|3            |HOWARD INC               |07877 Mark Bypass, Jacksonfurt, KY 08075             |howardinc@gmail.com             |84294493       |
|4            |JENKINS-GRAY             |909 Jennifer Plains Suite 93

In [6]:
print("=== PRODUITS ===")
produits_silver = produits \
    .dropDuplicates(["ProduitID"]) \
    .dropna(subset=["ProduitID", "NomProduit", "PrixUnitaire"]) \
    .filter(F.col("PrixUnitaire") > 0) \
    .withColumn("NomProduit",  F.trim(F.initcap(F.col("NomProduit")))) \
    .withColumn("Description", F.trim(F.col("Description"))) \
    .withColumn("PrixUnitaire", F.round(F.col("PrixUnitaire").cast("double"), 2))

print(f"Bronze : {produits.count()} lignes")
print(f"Silver : {produits_silver.count()} lignes")
produits_silver.show(5, truncate=False)

=== PRODUITS ===
Bronze : 100 lignes
Silver : 100 lignes
+---------+------------------+----------------------------------------------------------------------------------------------------------+------------+-------------+
|ProduitID|NomProduit        |Description                                                                                               |PrixUnitaire|FournisseurID|
+---------+------------------+----------------------------------------------------------------------------------------------------------+------------+-------------+
|1        |Samsung Galaxy S21|Featuring a 6.2-inch dynamic AMOLED display, triple rear cameras, and 8K video recording.                 |699.0       |4            |
|2        |Samsung Tv 55'    |55-inch 4K UHD Smart TV with HDR, Crystal Display, and built-in voice assistants.                         |129.0       |51           |
|3        |Nike Air Max      |Running shoes with Max Air cushioning, mesh uppers, and durable rubber outsole.         

In [7]:
print("=== VENTES ===")
ventes_silver = ventes \
    .dropDuplicates(["VenteID"]) \
    .dropna(subset=["VenteID", "DateVente", "ClientID", "ProduitID", "EmployeID"]) \
    .withColumn("DateVente",      F.to_date(F.col("DateVente"), "yyyy-MM-dd")) \
    .withColumn("QuantiteVendue", F.col("QuantiteVendue").cast("int")) \
    .withColumn("MontantTotal",   F.round(F.col("MontantTotal").cast("double"), 2)) \
    .filter(F.col("QuantiteVendue") > 0) \
    .filter(F.col("MontantTotal") > 0) \
    .withColumn("Annee", F.year(F.col("DateVente"))) \
    .withColumn("Mois",  F.month(F.col("DateVente")))

print(f"Bronze : {ventes.count()} lignes")
print(f"Silver : {ventes_silver.count()} lignes")
ventes_silver.show(5, truncate=False)

=== VENTES ===
Bronze : 100 lignes
Silver : 100 lignes
+-------+----------+--------+---------+---------+--------------+------------+-----+----+
|VenteID|DateVente |ClientID|ProduitID|EmployeID|QuantiteVendue|MontantTotal|Annee|Mois|
+-------+----------+--------+---------+---------+--------------+------------+-----+----+
|31     |2021-05-05|12      |14       |60       |1799          |1437401.0   |2021 |5   |
|85     |2020-08-30|46      |21       |30       |15000         |1275000.0   |2020 |8   |
|65     |2021-07-06|53      |29       |62       |1859          |741741.0    |2021 |7   |
|53     |2022-01-05|97      |41       |14       |4900          |1465100.0   |2022 |1   |
|78     |2023-06-23|25      |55       |24       |1920          |1342080.0   |2023 |6   |
+-------+----------+--------+---------+---------+--------------+------------+-----+----+
only showing top 5 rows



In [ ]:
import os

SILVER = "C:/Users/smagu/retail-data-platform/data/silver"

silver_tables = {
    "clients":      clients_silver,
    "employes":     employes_silver,
    "fournisseurs": fournisseurs_silver,
    "produits":     produits_silver,
    "ventes":       ventes_silver,
}

for nom, df in silver_tables.items():
    path = f"{SILVER}/{nom}"
    os.makedirs(path, exist_ok=True)
    # toPandas() + pyarrow : contourne le besoin de winutils.exe sur Windows
    df.toPandas().to_parquet(f"{path}/{nom}.parquet", index=False)
    print(f"Ecrit : {path}/{nom}.parquet")

print("\nToutes les tables Silver sont enregistrees !")

In [ ]:
import pandas as pd

print("=== VERIFICATION SILVER ===")
for nom in silver_tables.keys():
    path = f"{SILVER}/{nom}/{nom}.parquet"
    df_check = pd.read_parquet(path)
    print(f"\n{nom.upper()} — {len(df_check)} lignes, {len(df_check.columns)} colonnes : {list(df_check.columns)}")

print("\nVerification terminee.")